# Analisis Multivariat Ketahanan Pangan Jawa Tengah 2025
**Tahap Persiapan**: Import seluruh library yang dibutuhkan.

In [ ]:
!pip install factor-analyzer

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import chi2, skew
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

# Setup folder dan Palet Warna
os.makedirs('check', exist_ok=True)
okabe_ito = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
sns.set_palette(sns.color_palette(okabe_ito))
cmap_div = sns.diverging_palette(250, 30, s=90, l=50, center="light", as_cmap=True)

# Fungsi caption gambar
def add_caption(fig_obj, is_plotly=False):
    teks = "Sumber: BPS (diolah), 2025"
    if is_plotly:
        fig_obj.add_annotation(text=teks, xref="paper", yref="paper", x=1, y=-0.15, showarrow=False, font=dict(size=10))
    else:
        plt.figtext(0.99, 0.01, teks, horizontalalignment='right', fontsize=9, style='italic')

## TAHAP 1 - Inspeksi & Persiapan Data
Membaca data Excel, memvalidasi jumlah baris (harus 35 unit), cek missing value, duplikat, dan membuat variabel pengait (`kode_shp`) untuk Leaflet.

In [ ]:
import pandas as pd
# Jika kamu menggunakan Google Colab, library ini untuk memunculkan prompt download
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

# 1. Masukkan dictionary nama variabel dan ID File Google Drive-nya.
file_ids = {
    'kalori': '1KQmUnuNcrOSn4Cgixjp7EvhcqlRLk65O',
    'pct_kerja_tani': '1EiufSgrmBqGc_rCgmha9GBLfsUco0ux_',
    'pct_pengeluaran_makanan': '1ONYst9XaVc45kROT_ECG4VLobVfpTwg2',
    'produksi_cabai_kapita': '16nf29DrgLwLddV7VPKEAxfGmpJnD7kAR',
    'produksi_padi_kapita': '1EYuF23yqSGO_S6nWS0VNyCcC5qoKa7BS',
    'produksi_telur_kapita': '1N0IIX7Hl4lPrgG5o3WXguImdWEf4swqA',
    'produktivitas_padi': '1dGjI8LdZbm_-abYhhHdDv4Ryg9RzqMTW',
    'share_pdrb': '1R4SXtMuCW4AdUXs2aslxblofpNlJBjJI',
    'protein': '1ERQGeXBRwWYnylUE_7HdC7n3Bi9M2Qii'
}

df_final = pd.DataFrame()

print("Memulai proses load dan penggabungan data...\n")

# 2. Looping untuk load, bersihkan, dan gabung tiap file
for var_name, file_id in file_ids.items():
    print(f"Loading {var_name}...")

    # URL direct download Google Drive
    url = f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx'

    # Kita gunakan skiprows=1 karena berdasarkan gambar,
    # baris 1 adalah judul gabungan (pct_kerja_tani), dan baris 2 adalah header aslinya
    df_temp = pd.read_excel(url, skiprows=1)

    # Identifikasi nama kolom
    col_kode = df_temp.columns[0] # Harusnya 'kode_kab/kota'
    col_nama = df_temp.columns[1] # Harusnya 'nama_kab/kota'

    # Antisipasi tipe kolom tahun, apakah terdeteksi sebagai string '2025' atau integer 2025
    col_2025 = 2025 if 2025 in df_temp.columns else '2025'

    # Ambil kolom yang dibutuhkan (Kode, Nama, dan 2025)
    df_clean = df_temp[[col_kode, col_nama, col_2025]].copy()

    # Ubah nama kolom 2025 menjadi nama variabelnya
    df_clean.rename(columns={col_2025: var_name}, inplace=True)

    # Hapus baris 'Provinsi Jawa Tengah' (kode 3300) karena kita butuh 35 kab/kota saja
    df_clean = df_clean[df_clean[col_kode] != 3300]

    # Hapus baris kosong/NaN jika secara tidak sengaja terbaca di bawah tabel Excel
    df_clean = df_clean.dropna(subset=[col_kode])

    # Gabungkan ke DataFrame utama (df_final)
    if df_final.empty:
        df_final = df_clean
    else:
        # Gabungkan berdasarkan kode dan nama agar posisinya sejajar
        df_final = pd.merge(df_final, df_clean, on=[col_kode, col_nama], how='outer')

# 3. Merapikan nama kolom utama agar sesuai untuk tahap analisis sebelumnya
df_final.rename(columns={
    df_final.columns[0]: 'kode_bps',
    df_final.columns[1]: 'nama_wilayah'
}, inplace=True)

# Memastikan kode_bps berupa string (contoh: "3301") agar tidak ada nol di depan yang hilang
df_final['kode_bps'] = df_final['kode_bps'].astype(int).astype(str)

print("\n--- DATA BERHASIL DIGABUNGKAN ---")
display(df_final.head())

# 4. Menyimpan dan mendownload file hasil
# nama_file_keluaran = 'data_jateng_2025_bersih.xlsx'
# df_final.to_excel(nama_file_keluaran, index=False)

# if in_colab:
#     print(f"\nMendownload file {nama_file_keluaran} ke komputermu...")
#     files.download(nama_file_keluaran)
# else:
#     print(f"\nFile berhasil disimpan di foldermu dengan nama: {nama_file_keluaran}")

In [ ]:
# Mengambil data dari hasil gabungan Google Drive di sel sebelumnya
df = df_final.copy()

# 1. Cek dimensi
assert len(df) == 35, f"Fatal: Jumlah unit harus 35, terdeteksi {len(df)}."
print("Jumlah unit: 35 (Sesuai).")

# 2. Cek missing value dan duplikat
print("Missing values per kolom:\n", df.isnull().sum())
print("\nJumlah duplikat:", df.duplicated().sum())

# 3. Buat format kode_shp ("3301" -> "33.01") untuk Leaflet
df['kode_bps'] = df['kode_bps'].astype(str).str.zfill(4)
df['kode_shp'] = df['kode_bps'].apply(lambda x: f"{x[:2]}.{x[2:]}")

# --- PERBAIKAN DI SINI ---
# Mendefinisikan kolom variabel numerik dengan mengecualikan kolom identitas
vars_cols = [col for col in df.columns if col not in ['kode_bps', 'nama_wilayah', 'kode_shp']]
X_raw = df[vars_cols].copy()

# Pembersihan koma (menjadi titik) untuk format desimal Indonesia
for col in X_raw.columns:
    if X_raw[col].dtype == 'object':
        X_raw[col] = X_raw[col].astype(str).str.replace(',', '.').astype(float)

# Update dataframe utama agar seragam
df[vars_cols] = X_raw

print("\nDaftar variabel yang akan dianalisis (pastikan tidak ada kode_shp):")
print(vars_cols)

## TAHAP 2 - EDA & Pra-Pemrosesan
Mengecek korelasi ekstrem (|r| > 0.9), mendeteksi skewness tinggi (|skew| > 1) untuk transformasi log1p, dan melakukan standarisasi z-score.

In [ ]:
# 1. Transformasi Log1p untuk Skewness
skewness = X_raw.skew()
skewed_vars = skewness[abs(skewness) > 1].index.tolist()
X_transformed = X_raw.copy()
for col in skewed_vars:
    X_transformed[col] = np.log1p(X_transformed[col])

# 2. Z-Score Scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_transformed), columns=vars_cols)

# 3. Visualisasi Heatmap Korelasi
corr_matrix = X_raw.corr()
fig, ax = plt.subplots(figsize=(8, 6))
# Diurutkan agar variabel yang mirip mengelompok
col_linkage = linkage(X_scaled.T, method='ward')
col_order = leaves_list(col_linkage)
corr_ordered = corr_matrix.iloc[col_order, col_order]

sns.heatmap(corr_ordered, annot=True, cmap=cmap_div, center=0, fmt=".2f", ax=ax, square=True, annot_kws={"size": 8})
ax.set_title('Heatmap Korelasi Variabel (Pearson)')
add_caption(fig)
plt.savefig('check/heatmap_korelasi.png', dpi=300, bbox_inches='tight')
plt.show()

## TAHAP 3 & 4

In [ ]:
# =========================================================
# TEMPEL SEL INI TEPAT SETELAH TAHAP 2 EDA SELESAI
# =========================================================
# 1. Kunci Wilayah (Langkah 1)
df['kode_kabkota'] = df['kode_bps'].astype(str).str.replace('.', '', regex=False)

# 2. Definisikan 9 variabel valid (Mengabaikan variabel plot/lama)
valid_vars = [
    'produksi_padi_kapita', 'produksi_telur_kapita', 'produksi_cabai_kapita',
    'kalori', 'share_pdrb', 'pct_pengeluaran_makanan',
    'pct_kerja_tani', 'protein', 'produktivitas_padi'
]
vars_cols = [col for col in df.columns if col in valid_vars]
var_dict = {col: f"v{i+1}" for i, col in enumerate(vars_cols)}

# 3. Persiapan Data Versi B (Log1p -> Z-score)
X_mentah = df[vars_cols].copy()
X_mentah_skew = X_mentah.skew().to_dict()

X_B = X_mentah.copy()
log_vars = ['produksi_padi_kapita', 'produksi_telur_kapita', 'produksi_cabai_kapita']
for c in log_vars: X_B[c] = np.log1p(X_B[c])

scaler = StandardScaler()
X_z = pd.DataFrame(scaler.fit_transform(X_B), columns=vars_cols)
mean_B = X_B.mean().to_dict()
std_B = X_B.std(ddof=0).to_dict()

# 4. Diagnostik (KMO & Bartlett)
kmo_all, kmo_model = calculate_kmo(X_z)
chi2_bart, p_val_bart = calculate_bartlett_sphericity(X_z)
df_deg_bart = (len(vars_cols) * (len(vars_cols) - 1)) / 2

# 5. PCA 9 Komponen (Versi B)
pca = PCA(n_components=9, random_state=42)
scores = pca.fit_transform(X_z)
eigenvalues = pca.explained_variance_
variance_ratio = pca.explained_variance_ratio_ * 100
cum_variance = np.cumsum(variance_ratio)

loadings = pca.components_.T * np.sqrt(eigenvalues)
# Standarisasi Tanda PC agar deterministik
for i in range(9):
    if loadings[np.argmax(np.abs(loadings[:, i])), i] < 0:
        loadings[:, i] *= -1
        scores[:, i] *= -1

contributions = (loadings**2 / eigenvalues) * 100
cos2 = loadings**2
print("✅ PCA Versi B Selesai dihitung.")

In [ ]:
# =========================================================
# 1. TABEL PCA.EIGEN (Ringkasan Eigenvalue & Komponen)
# =========================================================
df_eigen = pd.DataFrame({
    'Komponen': [f"PC{i+1}" for i in range(len(eigenvalues))],
    'Eigenvalue': eigenvalues,
    '% Varians': variance_ratio,
    '% Kumulatif Varians': cum_variance
})

print("=== TABEL PCA.EIGEN ===")
print(df_eigen.to_string(index=False))
print("\n" + "="*50 + "\n")

# =========================================================
# 2. TABEL PCA.STANDARISASI (Parameter Transformasi & Z-Score)
# =========================================================
df_standarisasi = pd.DataFrame({
    'Variabel': vars_cols,
    'Alias': [var_dict[col] for col in vars_cols],
    'Skewness_Mentah': [X_mentah_skew[col] for col in vars_cols],
    'Is_Log1p': [col in log_vars for col in vars_cols],
    'Mean_PascaLog': [mean_B[col] for col in vars_cols],
    'Std_PascaLog': [std_B[col] for col in vars_cols]
})

print("=== TABEL PCA.STANDARISASI ===")
print(df_standarisasi.to_string(index=False))

## TAHAP 5 - 6

In [ ]:
# 1. PENCILAN (Mahalanobis PC1-PC3)
pc_std = np.std(scores[:, :3], axis=0, ddof=1)
mahala = np.sum((scores[:, :3] / pc_std)**2, axis=1)
threshold = chi2.ppf(0.975, df=3)

df['mahalanobis'] = mahala
df['is_outlier'] = mahala > threshold
df['jarak_rank'] = df['mahalanobis'].rank(ascending=False).astype(int)

# 2. EVALUASI K-MEANS
sil_scores, km_models = {}, {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_z)
    sil_scores[k] = silhouette_score(X_z, km.labels_)
    km_models[k] = km

# 3. KONFIGURASI KLASTER FINAL (Bisa kamu ubah nanti)
KONFIG_KLASTER = {
    "k": 3,
    "labels": {
        1: "Tinggi Konsumsi, Rendah Agraris (Perkotaan)",
        2: "Sentra Produksi Pangan (Lumbung)",
        3: "Tengah-Tengah / Transisi"
    }
}
final_labels = km_models[KONFIG_KLASTER["k"]].labels_
counts_final = pd.Series(final_labels).value_counts()
map_final = {old: new+1 for new, old in enumerate(counts_final.index)}
df['klaster_final'] = [map_final[l] for l in final_labels]

# 4. KORELASI & URUTAN HEATMAP
corr_matrix = X_z.corr().values
dist_matrix = 1 - np.abs(corr_matrix)
np.fill_diagonal(dist_matrix, 0)
from scipy.spatial.distance import squareform
link = linkage(squareform(dist_matrix, checks=False), method='average')
var_order = [vars_cols[i] for i in leaves_list(link).tolist()]
print(f"✅ Klastering & Urutan Korelasi selesai. Silhouette K={KONFIG_KLASTER['k']}: {sil_scores[KONFIG_KLASTER['k']]:.4f}")

## TAHAP 7 - Ekspor ke JSON & CSV
Menyatukan semua hasil analisis (skor, loading, urutan heatmap, metrik diagnostik) ke dalam satu file JSON untuk ditarik langsung oleh React, serta satu CSV datar untuk inspeksi manual.

In [ ]:
import json, os

def clean_nan(val):
    return None if pd.isna(val) or np.isinf(val) else float(val)

# Pastikan folder public/data/ ada
os.makedirs('public/data', exist_ok=True)

output_json = {
    "catatan_data": {
        "keterangan": "PCA cross-section 35 Kab/Kota Jateng 2025. Cabai bernilai 0 di Kota Pekalongan tetap di-log1p.",
        "kmo_overall": round(kmo_model, 3), "rasio_obs_var": "35:9"
    },
    "variabel": [{"id": var_dict[c], "nama": c, "satuan": "", "arah": "positif", "urutan": i,
                  "pca_loading": np.round(loadings[i], 4).tolist(),
                  "pca_kontribusi": np.round(contributions[i], 4).tolist(),
                  "pca_cos2": np.round(cos2[i], 4).tolist()} for i, c in enumerate(vars_cols)],
    "diagnostik": {
        "kmo": round(kmo_model, 3),
        "bartlett": {"chi2": round(chi2_bart, 2), "df": int(df_deg_bart), "p_value": clean_nan(p_val_bart)},
        "msa_per_variabel": {c: round(kmo_all[i], 3) for i, c in enumerate(vars_cols)}
    },
    "pca": {
        "transformasi": {"log1p": log_vars, "alasan": "skewness > 0,8"},
        "standarisasi": "Z-score (ddof=0)",
        "skewness_mentah": {c: round(X_mentah_skew[c], 3) for c in vars_cols},
        "zscore_mean": {c: clean_nan(mean_B[c]) for c in vars_cols},
        "zscore_std": {c: clean_nan(std_B[c]) for c in vars_cols},
        "eigen": {"eigenvalue": np.round(eigenvalues, 4).tolist(), "varians_persen": np.round(variance_ratio, 4).tolist(), "kumulatif_persen": np.round(cum_variance, 4).tolist()}
    },
    "pencilan_meta": {
        "kriteria": "Jarak Mahalanobis pada PC1-PC3", "ambang_chi2_975": round(threshold, 3), "df": 3,
        "top_5": df.nlargest(5, 'mahalanobis')[['kode_kabkota', 'nama_wilayah', 'mahalanobis']].to_dict('records')
    },
    "klaster_meta": {
        "metode": "K-Means", "input_data": "Z-score (Versi B)", "k": KONFIG_KLASTER["k"],
        "silhouette": round(sil_scores[KONFIG_KLASTER["k"]], 4), "profil": []
    },
    "korelasi": {"urutan_variabel_hierarkis": var_order, "matriks": np.round(corr_matrix, 4).tolist()},
    "units": []
}

for k_id in range(1, KONFIG_KLASTER["k"] + 1):
    mask = df['klaster_final'] == k_id
    output_json["klaster_meta"]["profil"].append({
        "id_klaster": k_id, "nama": KONFIG_KLASTER["labels"].get(k_id, f"Klaster {k_id}"),
        "jumlah_anggota": int(mask.sum()), "rata_rata_z": {c: clean_nan(X_z[mask][c].mean()) for c in vars_cols}
    })

for i, row in df.iterrows():
    output_json['units'].append({
        "kode_kabkota": row['kode_kabkota'], "nama": row['nama_wilayah'],
        "nilai_mentah": {var_dict[c]: clean_nan(X_mentah.iloc[i][c]) for c in vars_cols},
        "nilai_z": {var_dict[c]: clean_nan(X_z.iloc[i][c]) for c in vars_cols},
        "skor_pc": np.round(scores[i], 4).tolist(), "klaster": int(row['klaster_final']),
        "pencilan": {"is_outlier": bool(row['is_outlier']), "jarak_mahalanobis": clean_nan(row['mahalanobis']), "peringkat": int(row['jarak_rank'])}
    })

with open('public/data/output_pca.json', 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2, allow_nan=False)

print(f"📦 Ukuran file JSON: {os.path.getsize('public/data/output_pca.json') / 1024:.1f} KB")
print("✅ JSON sukses diekspor dan siap dipakai di React!")

In [ ]:
import json

with open('public/data/output_pca.json', 'r', encoding='utf-8') as f:
    cek_json = json.load(f)

print("=== KUNCI TINGKAT ATAS ===")
print(list(cek_json.keys()))

print("\n=== CONTOH WILAYAH LENGKAP (3301) ===")
wilayah_3301 = next((u for u in cek_json.get('units', []) if u.get('kode_kabkota') == "3301"), None)
print(json.dumps(wilayah_3301, indent=2))

print("\n=== BLOK CATATAN DATA (UTUH) ===")
print(json.dumps(cek_json.get('catatan_data'), indent=2))

print("\n=== BLOK DIAGNOSTIK (UTUH) ===")
print(json.dumps(cek_json.get('diagnostik'), indent=2))

print("\n=== BLOK PENCILAN META (UTUH) ===")
print(json.dumps(cek_json.get('pencilan_meta'), indent=2))

print("\n=== BLOK KLASTER META (UTUH) ===")
print(json.dumps(cek_json.get('klaster_meta'), indent=2))

print("\n=== BLOK VARIABEL (UTUH) ===")
print(json.dumps(cek_json.get('variabel'), indent=2))

print("\n=== STRUKTUR BLOK LAIN ===")
# Blok PCA
pca_block = cek_json.get('pca', {})
print("1. BLOK 'pca'")
print("   Tipe Data:", type(pca_block).__name__)
print("   Kunci:", list(pca_block.keys()))
print("   Contoh nilai 'transformasi':", json.dumps(pca_block.get('transformasi'), indent=2))

# Blok Korelasi
korelasi_block = cek_json.get('korelasi', {})
matriks = korelasi_block.get('matriks', [])
print("\n2. BLOK 'korelasi'")
print("   Tipe Data:", type(korelasi_block).__name__)
print("   Kunci:", list(korelasi_block.keys()))
print(f"   Panjang 'urutan_variabel_hierarkis': {len(korelasi_block.get('urutan_variabel_hierarkis', []))}")
print(f"   Dimensi 'matriks': {len(matriks)} baris x {len(matriks[0]) if matriks else 0} kolom")
print("   Contoh 1 baris pertama matriks:", json.dumps(matriks[0] if matriks else [], indent=2))

# Blok Units (Struktur Umum)
units_block = cek_json.get('units', [])
print("\n3. BLOK 'units'")
print("   Tipe Data:", type(units_block).__name__)
print("   Panjang List:", len(units_block))
print("   Kunci pada elemen pertama:", list(units_block[0].keys()) if units_block else [])